# TP 11 · Clustering — Cómo le fue al curso

PedidosYa Growth: una sola campaña dirigida, ¿a quién? Todo agregado y anónimo. Gráficos interactivos (mouse, zoom, ▶).

In [1]:
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
import plotly.io as pio
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
W=920; pio.templates.default='plotly_white'
VERDE,AMBAR,ROJO,AZUL,GRIS='#55A868','#E1A100','#C44E52','#4C72B0','#9aa'
def fijar(f,h):
    f.update_layout(width=W,height=h,autosize=False,margin=dict(t=70,b=45,l=60,r=30)); return f
print('ok')

ok


## 1 · El camino (CRISP-DM)

Seis fases; la decisión es la última. Pasá el mouse por cada una.

In [2]:
fases=['1·Negocio','2·Datos','3·Preparación','4·Modelado','5·Evaluación','6·Decisión']
mapa=['¿a quién apuntar?','¿confío en los datos?','error vs señal','¿cuántos grupos?','¿quién es cada uno?','el cupón, ¿a quién?']
ang=np.linspace(90,90-360,6,endpoint=False)*np.pi/180; x,y=np.cos(ang),np.sin(ang)
fig=go.Figure()
for i in range(6):
    j=(i+1)%6
    fig.add_annotation(x=x[j]*0.8,y=y[j]*0.8,ax=x[i]*0.8,ay=y[i]*0.8,xref='x',yref='y',axref='x',ayref='y',showarrow=True,arrowhead=3,arrowwidth=2,arrowcolor='#94a3b8')
fig.add_trace(go.Scatter(x=x,y=y,mode='markers+text',text=fases,textposition='middle center',marker=dict(size=92,color=[AZUL,AZUL,AZUL,ROJO,VERDE,'#7d3c98']),textfont=dict(color='white',size=11),hovertext=mapa,hoverinfo='text'))
fig.add_annotation(x=0,y=0,text='<b>CRISP-DM</b>',showarrow=False,font=dict(size=17))
fig.update_layout(title='El recorrido del TP',showlegend=False,xaxis=dict(visible=False,range=[-1.45,1.45]),yaxis=dict(visible=False,range=[-1.4,1.4],scaleanchor='x'))
fijar(fig,500); fig.show()

## 2 · Cómo les fue

Promedio 8.5. Todos entre 8 y 10.

In [3]:
vals=[10,8,1]
fig=go.Figure(go.Bar(x=['8','9','10'],y=vals,marker_color=[AZUL,VERDE,'#2e7d32'],text=vals,textposition='outside'))
fig.update_layout(title='Distribución de notas · promedio 8.53 · 19 entregas',xaxis_title='nota',yaxis_title='grupos')
fijar(fig,380); fig.show()

## 3 · La técnica la dominaron

Negocio, lo más fuerte; evaluación, lo más flojo. Ahí se jugó el TP.

In [4]:
sec=["Negocio", "Datos/EDA", "Preparación", "K-Means", "Evaluación"]; prom=[9.08, 8.34, 8.03, 8.5, 7.95]
mx=max(prom); mn=min(prom)
col=[VERDE if v==mx else (ROJO if v==mn else '#8aa0b8') for v in prom]
fig=go.Figure(go.Bar(x=sec,y=prom,marker_color=col,text=[f'{v:.1f}' for v in prom],textposition='outside'))
fig.update_layout(title='Promedio por sección de la consigna',yaxis=dict(range=[0,10],title='promedio')); fijar(fig,400); fig.show()

## 4 · El algoritmo te da K, vos elegís

El silhouette pedía k=2; en k=4 aparecen los dormidos. ▶

In [5]:
df=pd.read_csv('usuarios_pedidosya.csv')
gastos=['gasto_restaurantes','gasto_mercados','gasto_bebidas','gasto_farmacia','gasto_kioscos','gasto_premium']
d=df.drop_duplicates('usuario_id').copy(); d['edad']=2026-d['anio_nacimiento']; d=d[d['edad']<=100]
d=d[(d[gastos]>=0).all(axis=1)]; d['ingreso_estimado']=d['ingreso_estimado'].fillna(d['ingreso_estimado'].median())
d['gasto_total']=d[gastos].sum(axis=1)
feat=gastos+['ingreso_estimado','dias_desde_ultimo_pedido']; X=StandardScaler().fit_transform(d[feat])
fr=[]
for k in [2,3,4,5,6]:
    lab=KMeans(n_clusters=k,random_state=42,n_init=10).fit_predict(X)
    t=d[['gasto_total','dias_desde_ultimo_pedido']].copy(); t['K']=k; t['grupo']=lab.astype(str); fr.append(t)
anim=pd.concat(fr)
fig=px.scatter(anim,x='dias_desde_ultimo_pedido',y='gasto_total',color='grupo',animation_frame='K',range_x=[-3,105],range_y=[-50000,anim['gasto_total'].max()*1.05],opacity=0.55,color_discrete_sequence=px.colors.qualitative.Set2,labels={'dias_desde_ultimo_pedido':'días sin pedir','gasto_total':'gasto 2 años'},title='K-Means para distinto K (▶)')
fig.update_traces(marker=dict(size=5)); fig.layout.sliders[0].currentvalue.prefix='K = '
fijar(fig,500); fig.show()

## 5 · ¿A quién el cupón?

El resuelto va a los dormidos. El curso se partió casi en tres.

In [6]:
labels=['Dormidos (incremental)','Intermedio (mirones/activar)','Alto valor (plata quemada)']
vals=[7,5,7]
fig=go.Figure(go.Bar(x=labels,y=vals,marker_color=[VERDE,AMBAR,ROJO],text=vals,textposition='outside'))
fig.update_layout(title='¿A qué segmento dirigieron la campaña? (19 entregas)',yaxis_title='grupos',
    annotations=[dict(x=2,y=vals[2]+0.6,text='= los que iban a comprar igual',showarrow=False,font=dict(color=ROJO,size=11))])
fijar(fig,420); fig.show()

**La paradoja:** razonan la incrementalidad para decidir a quién *no* mandar, pero 7 igual eligieron el alto valor (el que compraba igual).

## 6 · Por qué los dormidos

La campaña masiva la canjearon los que ya compraban. El dormido casi no responde, pero ahí el canje es incremental.

In [7]:
d['cluster']=KMeans(n_clusters=4,random_state=42,n_init=10).fit_predict(X)
nombres={0:'full-canasta',1:'antojo',2:'dormidos',3:'gourmet'}; d['segmento']=d['cluster'].map(nombres)
tasas=(d.groupby('segmento')['respuesta_ultima_campania'].mean()*100).reindex(['full-canasta','gourmet','antojo','dormidos'])
fig=go.Figure(go.Bar(x=tasas.index,y=tasas.values,marker_color=[GRIS,GRIS,GRIS,VERDE],text=[f'{v:.0f}%' for v in tasas.values],textposition='outside'))
fig.update_layout(title='% que respondió la última campaña masiva',yaxis_title='%'); fijar(fig,420); fig.show()

## 7 · Lo que aportaron

- La paradoja de los mirones: los que más navegan, menos compran (r = -0,5).
- DBSCAN leído como que los heavy users son ruido, no público del cupón.
- Imputaciones más finas (mediana por ciudad) y nombres de segmento propios.

## 8 · Qué se logró

In [8]:
logros={'Escaló los datos':19,'Eligió K=4':13,'Hizo el bonus':14,'Apuntó a los dormidos':7}
it=list(logros.keys()); vl=list(logros.values()); col=[VERDE if v>=14 else (AMBAR if v>=8 else ROJO) for v in vl]
fig=go.Figure(go.Bar(x=vl,y=it,orientation='h',marker_color=col,text=vl,textposition='outside'))
fig.update_layout(title='Qué logró el curso (de 19 entregas)',xaxis=dict(range=[0,20],title='grupos')); fig.update_yaxes(autorange='reversed'); fijar(fig,360); fig.show()

## Para cerrar

La pregunta para llevarse: **¿este pedido iba a pasar igual?** Si sí, no es incremental.

La métrica acota; el analista decide. El cupón vale donde cambia la conducta, no donde el pedido pasaba igual.